# Reporting-Delay Chain Ladder — Clean Reproducible Notebook

This notebook mirrors the public-safe pipeline in `src/reporting_delay_chain_ladder.py`.
Raw claim data is intentionally not included in the repository.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
sys.path.append(str(ROOT / 'src'))
import reporting_delay_chain_ladder as cl


## Run the pipeline

Place the confidential workbook at `data/Dataset.xlsx` and, if applicable, a verified local repair file at `config/manual_repairs.csv`.


In [ ]:
claims = cl.load_claims(ROOT / 'data' / 'Dataset.xlsx')
claims = cl.parse_dates(claims)
claims = cl.remove_exact_duplicates(claims)
claims = cl.apply_manual_repairs(claims, ROOT / 'config' / 'manual_repairs.csv')
mask = cl.valid_model_mask(claims)
model_data = claims.loc[mask].copy()
excluded = claims.loc[~mask].copy()
triangle_data, incremental, cumulative, max_dev = cl.construct_triangles(model_data)
factors = cl.calculate_factors(cumulative, max_dev)
cdf = cl.calculate_cdfs(factors, max_dev)
results = cl.project_ibnr(cumulative, cdf, max_dev)
results


In [ ]:
reported = results['Reported at Valuation'].sum()
ultimate = results['Projected Ultimate'].sum()
ibnr = results['IBNR'].sum()
print(f'Reported: {reported:,.2f}')
print(f'Ultimate: {ultimate:,.2f}')
print(f'IBNR: {ibnr:,.2f}')
